# PaliGemma from scratch: checking against the real weights

We load the `google/paligemma-3b-pt-224` weights into our own model and compare it with Hugging Face.

**Before you start**
1. **GPU:** Runtime → Change runtime type → pick a GPU. The free T4 is enough (it runs bf16 slowly but it runs), an A100 is much faster.
2. **Hugging Face license:** accept the license on the [google/paligemma-3b-pt-224](https://huggingface.co/google/paligemma-3b-pt-224) page. Accepting PaliGemma 2 is not enough, this is PaliGemma 1.
3. **Hugging Face token:** in Colab, left sidebar 🔑 Secrets → add it as `HF_TOKEN` and enable "Notebook access". Otherwise a cell opens a login prompt.
4. **Upload to Drive:** copy the repo's `src/` folder to `Drive/paligemma2/src/`. Put the images you want to try in `Drive/paligemma2/images/` (`.jpg`/`.png`, directly in the folder, no subfolders). If `src/` changes, update the copy on Drive and restart the session: Python does not re-read a module it has already imported.

**Memory plan (T4: 15 GB GPU, ~12 GB RAM):** the model is ~6 GB in bf16. We load the HF model on the GPU and compute the references,
copy its weights to the CPU and delete it from the GPU, then build our own model and load the same weights.
That way only one model is on the GPU at a time.

**What we check**
- Does our processor give the same `input_ids` and `pixel_values` as HF?
- Are the logits for the same input the same as HF's?
- Is greedy generation token-for-token the same as HF `generate(do_sample=False)`?
- What does it produce on your own images?
- If the logits differ, where does the difference come from? (diagnosis)
- KV cache: is cached generation the same as the uncached path and HF, and how fast is it per token?

In [ ]:
import sys, time
from pathlib import Path

import torch

assert torch.cuda.is_available(), "No GPU: set Runtime → Change runtime type → a GPU"
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

# Folders on Drive. Change the path to match your layout
DRIVE_ROOT = Path("/content/drive/MyDrive/paligemma2")
SRC_DIR = DRIVE_ROOT / "src"
IMAGES_DIR = DRIVE_ROOT / "images"

MODEL_ID = "google/paligemma-3b-pt-224"
PROMPT = "caption en"        # the pt model was trained with task prefixes: "caption en", "ocr", "detect cat", "answer en ..."
MAX_NEW_TOKENS = 20

assert (SRC_DIR / "paligemma.py").exists(), f"no paligemma.py in {SRC_DIR}"
sys.path.insert(0, str(SRC_DIR))

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN read from Colab Secrets")
except Exception:
    from huggingface_hub import login
    login()

import transformers
print("transformers", transformers.__version__, "| torch", torch.__version__)

## Reference: the HF model

We load the HF model in bf16 with eager attention (the same math as ours). To make the prefix bidirectional we pass
`token_type_ids = 0`. Without it HF processes image + prompt causally and does not compute the same thing as we do
(on a small random model the logits differ by ~8–10, see `checks/check_paligemma.py`).

In [ ]:
from PIL import Image
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration

# Images directly inside the folder (subfolders are ignored). jpg, jpeg and png, any case
image_paths = sorted(p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
assert image_paths, f"no images in {IMAGES_DIR}. Check the folder path and contents: " + str([p.name for p in IMAGES_DIR.iterdir()][:10])
image = Image.open(image_paths[0]).convert("RGB")
print("Test image:", image_paths[0].name, image.size)

hf_processor = AutoProcessor.from_pretrained(MODEL_ID)
hf_inputs = hf_processor(text=PROMPT, images=image, return_tensors="pt")
hf_pixel_values_fp32 = hf_inputs["pixel_values"].clone()      # float32 copy for the processor comparison
T = hf_inputs["input_ids"].shape[1]
print("T =", T, "(256 image tokens + <bos> + prompt + \\n)")
print("HF processor outputs:", list(hf_inputs.keys()))

In [ ]:
t0 = time.time()
hf_model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, attn_implementation="eager"
).to("cuda").eval()
print(f"HF model loaded: {time.time() - t0:.0f} s, GPU memory {torch.cuda.memory_allocated() / 1e9:.1f} GB")

input_ids = hf_inputs["input_ids"].to("cuda")
attention_mask = hf_inputs["attention_mask"].to("cuda")
pixel_values = hf_inputs["pixel_values"].to("cuda", torch.bfloat16)
token_type_ids = torch.zeros_like(input_ids)                    # hepsi prefix

with torch.no_grad():
    ref_logits = hf_model(input_ids=input_ids, pixel_values=pixel_values, attention_mask=attention_mask,
                          token_type_ids=token_type_ids).logits.float().cpu()      # [1, T, 257216]
    ref_out = hf_model.generate(input_ids=input_ids, pixel_values=pixel_values, attention_mask=attention_mask,
                                token_type_ids=token_type_ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                eos_token_id=hf_processor.tokenizer.eos_token_id,    # the same eos we pass to our generate
                                pad_token_id=hf_processor.tokenizer.pad_token_id)
ref_tokens = ref_out[:, T:].cpu()                               # HF also returns the prompt
print("HF logit'leri:", tuple(ref_logits.shape))
print("HF output    :", repr(hf_processor.decode(ref_tokens[0], skip_special_tokens=True)))

In [ ]:
# Move the weights to CPU (keys in HF layout, load_hf_weights maps them to ours), free the GPU
hf_sd = {k: v.detach().cpu() for k, v in hf_model.state_dict().items()}
print("HF state_dict key count:", len(hf_sd))
print("first 3:", list(hf_sd)[:3])
eos_token_id = hf_processor.tokenizer.eos_token_id
del hf_model
import gc; gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## Our model

The real config (the `PaliGemmaConfig()` defaults are the values from `paligemma-3b-pt-224`'s config.json). We build the model directly on the GPU in bf16:
building it in float32 on the CPU first would not fit in 12 GB of RAM.

`load_hf_weights` **raises** if the keys do not match one to one. The error message lists the keys that did not match.

In [ ]:
import paligemma, generate, processing

config = paligemma.PaliGemmaConfig()
torch.set_default_dtype(torch.bfloat16)
with torch.device("cuda"):
    model = paligemma.PaliGemmaForConditionalGeneration(config)
torch.set_default_dtype(torch.float32)

paligemma.load_hf_weights(model, hf_sd)
model.eval()
del hf_sd; gc.collect()
n_params = sum(p.numel() for p in model.parameters())
print(f"Our model is ready: {n_params / 1e9:.2f} B parameters, GPU memory {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print("weight tying:", model.language_model.lm_head.weight is model.language_model.model.embed_tokens.weight)

## Does our processor match HF?

In [ ]:
our_processor = processing.PaliGemmaProcessor(
    tokenizer=hf_processor.tokenizer, num_image_tokens=config.text_config.num_image_tokens,
    image_size=config.vision_config.image_size,
)
ours = our_processor(text=[PROMPT], images=[image])

ids_equal = torch.equal(ours["input_ids"], hf_inputs["input_ids"])
pix_diff = (ours["pixel_values"].float() - hf_pixel_values_fp32).abs().max().item()
print(f"input_ids identical: {ids_equal} ({tuple(ours['input_ids'].shape)})")
print(f"pixel_values max diff: {pix_diff:.2e}")

## Logit comparison

The same weights with the same order of operations give bit-identical logits, even in bf16. If two implementations do the same math in a different order,
small differences appear and pile up over 45 layers (27 SigLIP + 18 Gemma). So besides `allclose` we also look at:
the largest/mean difference, whether `argmax` agrees at every position, and whether the top-5 agrees at the last position. The real test is the next cell: are the generated tokens the same?

In [ ]:
with torch.no_grad():
    logits = model(input_ids, pixel_values, attention_mask).float().cpu()    # [1, T, 257216]

diff = (logits - ref_logits).abs()
print(f"shape: ours {tuple(logits.shape)}, HF {tuple(ref_logits.shape)}")
print(f"max diff {diff.max():.3f}, mean diff {diff.mean():.5f}, logit scale (mean |HF|) {ref_logits.abs().mean():.2f}")
print(f"allclose(atol=1e-2): {torch.allclose(logits, ref_logits, atol=1e-2)}")

agree = (logits.argmax(-1) == ref_logits.argmax(-1)).float()
print(f"share of positions with the same argmax: {agree.mean():.4f}  (last position: {bool(agree[0, -1])})")
top5_ours, top5_hf = logits[0, -1].topk(5).indices.tolist(), ref_logits[0, -1].topk(5).indices.tolist()
print("last position top-5 ours:", top5_ours)
print("last position top-5 HF  :", top5_hf)

## Greedy generation: token for token against HF

Our `generate` uses the KV cache by default. The comparison with the uncached path and the timing are in the KV cache part at the end.
In bf16, if two tokens have very close logits their order can flip. If that happens we print the first differing position and both candidates.

In [ ]:
t0 = time.time()
our_tokens = generate.generate(model, input_ids, pixel_values, attention_mask, MAX_NEW_TOKENS, eos_token_id).cpu()
print(f"generation time: {time.time() - t0:.1f} s ({our_tokens.shape[1]} tokens, with KV cache)")

n = min(our_tokens.shape[1], ref_tokens.shape[1])
same = torch.equal(our_tokens[:, :n], ref_tokens[:, :n]) and our_tokens.shape[1] == ref_tokens.shape[1]
print("ours:", repr(hf_processor.decode(our_tokens[0], skip_special_tokens=True)))
print("HF  :", repr(hf_processor.decode(ref_tokens[0], skip_special_tokens=True)))
print("identical tokens:", same)
if not same:
    mismatch = (our_tokens[0, :n] != ref_tokens[0, :n]).nonzero()
    if len(mismatch):
        i = mismatch[0].item()
        print(f"first difference at token {i}: ours {our_tokens[0, i].item()} ({hf_processor.decode(our_tokens[0, i])!r}), "
              f"HF {ref_tokens[0, i].item()} ({hf_processor.decode(ref_tokens[0, i])!r})")

## On your own images

The `pt` model works with task prefixes: `ocr` reads text, `caption en` describes in English, `answer en <question>` answers a question.

In [ ]:
from IPython.display import display

PROMPTS = ["caption en", "ocr"]
for path in image_paths[:3]:
    img = Image.open(path).convert("RGB")
    display(img.resize((img.width * 240 // img.height, 240)))
    for prompt in PROMPTS:
        enc = our_processor(text=[prompt], images=[img])
        t0 = time.time()
        out = generate.generate(model, enc["input_ids"].to("cuda"), enc["pixel_values"].to("cuda", torch.bfloat16),
                                enc["attention_mask"].to("cuda"), MAX_NEW_TOKENS, eos_token_id).cpu()
        print(f"[{path.name}] {prompt!r} -> {hf_processor.decode(out[0], skip_special_tokens=True)!r}  ({time.time() - t0:.0f} s)")

## Diagnosis: is a logit difference bf16 noise or a real mismatch?

If the logit comparison shows a difference (`allclose` fails), three steps tell bf16 rounding apart from a real difference in the code.
If the difference is zero these cells show zeros too and can be skipped.

- **Where the difference is** (without reloading HF): where is it concentrated, and was HF already unsure where argmax disagrees?
- **Relative error layer by layer** (HF is reloaded, both models on the GPU): does the error grow smoothly or jump somewhere?
- **fp32 test** (on CPU): with the same real weights, do the whole SigLIP and a few Gemma layers match HF in **float32**?
  If so the code is right and what remains is bf16 rounding.

In [ ]:
# Where the difference is. Loads no new model, uses `logits` and `ref_logits` from the logit comparison
n_img = config.text_config.num_image_tokens
diff = (logits - ref_logits).abs()                                   # [1, T, V]
agree = (logits.argmax(-1) == ref_logits.argmax(-1))[0]              # [T]
top2 = ref_logits[0].topk(2, dim=-1).values
margin = top2[:, 0] - top2[:, 1]                                     # gap between HF's 1st and 2nd candidate: if small, HF itself is unsure

print(f"{'region':20s} {'positions':>6s} {'mean |diff|':>11s} {'same argmax':>12s} {'HF top1-top2':>13s}")
for name, sl in [("image positions", slice(0, n_img)), ("<bos> + prompt + \\n", slice(n_img, T))]:
    print(f"{name:20s} {sl.stop - sl.start:6d} {diff[0, sl].mean():11.3f} {agree[sl].float().mean():12.3f} {margin[sl].mean():13.2f}")

dis = ~agree
print(f"\n{int(dis[:n_img].sum())} of the {int(dis.sum())} positions where argmax differs are image positions")
if dis.any() and agree.any():
    print(f"HF top1-top2 gap: mean {margin[dis].mean():.2f} where argmax differs, {margin[agree].mean():.2f} where it agrees")
    print("(if clearly smaller where argmax differs: the differences sit where HF was already unsure, and bf16 noise flips the order)")
per_pos = diff[0].mean(-1)
print("\n5 positions with the largest mean difference:", [(int(i), round(float(per_pos[i]), 2), "image" if i < n_img else "text") for i in per_pos.topk(min(5, T)).indices])
print(f"last position: mean |diff| {per_pos[-1]:.3f}, HF top1-top2 {margin[-1]:.2f}, same argmax: {bool(agree[-1])}")

### Relative error layer by layer

We load HF once more (from the cache, much faster than the first time). While both models run on the same input, we record every block's output with a `forward hook` and compute
the **relative error** `‖ours − HF‖ / ‖HF‖`. Both use the same `pixel_values`, so the 1/255-level pixel difference in the processor does not affect this test.

How to read it: a single bf16 operation rounds by ~4e-3 relative. After 45 layers a 1–5% relative error is normal. If the error grows **smoothly** it is noise; if it
**jumps** at one layer (marked `<<< jump`), that layer has a real difference.

In [ ]:
# Load HF again and capture both models' intermediate outputs with hooks
hf2 = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, attn_implementation="eager"
).to("cuda").eval()
device = next(model.parameters()).device
print(f"GPU memory (two models): {torch.cuda.memory_allocated() / 1e9:.1f} GB" if device.type == "cuda" else "cpu")


def first_attr(obj, *paths):
    # names change between transformers versions (5.x: model.vision_tower, 4.x: vision_tower)
    for path in paths:
        o = obj
        try:
            for part in path.split("."):
                o = getattr(o, part)
            return o
        except AttributeError:
            continue
    raise AttributeError(f"none of these exist: {paths}")


def as_tensor(x):
    if isinstance(x, tuple):
        x = x[0]
    if hasattr(x, "last_hidden_state"):
        x = x.last_hidden_state
    return x


hf_vt = first_attr(hf2, "model.vision_tower", "vision_tower")
hf_vt_in = getattr(hf_vt, "vision_model", hf_vt)
hf_proj = first_attr(hf2, "model.multi_modal_projector", "multi_modal_projector")
hf_lm = first_attr(hf2, "model.language_model", "language_model.model")
our_vt = model.vision_tower
our_vt_in = our_vt.vision_model
our_lm = model.language_model.model

# RoPE frequencies must be float32 and identical on both sides. If HF rounded them to bf16 (a .to(bfloat16) can do that)
# every Gemma layer would show a small difference
hf_inv, our_inv = first_attr(hf_lm, "rotary_emb").inv_freq, our_lm.rotary_emb.inv_freq
print(f"RoPE inv_freq: HF {hf_inv.dtype}, ours {our_inv.dtype}, max diff {(hf_inv.float().cpu() - our_inv.float().cpu()).abs().max().item():.2e}")

pairs = [("SigLIP embeddings", hf_vt_in.embeddings, our_vt_in.embeddings)]
pairs += [(f"SigLIP layer {i:2d}", h, o) for i, (h, o) in enumerate(zip(hf_vt_in.encoder.layers, our_vt_in.encoder.layers))]
pairs += [("SigLIP output", hf_vt, our_vt), ("projector", hf_proj, model.multi_modal_projector)]
pairs += [(f"Gemma layer {i:2d}", h, o) for i, (h, o) in enumerate(zip(hf_lm.layers, our_lm.layers))]
pairs += [("Gemma final norm", hf_lm.norm, our_lm.norm)]

hf_acts, our_acts, handles = {}, {}, []


def save_output(store, name):
    # A forward hook must return None: returning a value REPLACES the module's output with it (setdefault would return one)
    def hook(module, args, output):
        store.setdefault(name, as_tensor(output).detach().float().cpu())
    return hook


for name, h_mod, o_mod in pairs:
    handles.append(h_mod.register_forward_hook(save_output(hf_acts, name)))
    handles.append(o_mod.register_forward_hook(save_output(our_acts, name)))


def grab_first_input(store, key):
    # The tensor entering Gemma's first layer (the input of the fp32 test)
    def hook(module, args, kwargs):
        store.setdefault(key, (args[0] if args else kwargs["hidden_states"]).detach().float().cpu())
    return hook


handles.append(hf_lm.layers[0].register_forward_pre_hook(grab_first_input(hf_acts, "gemma_in0"), with_kwargs=True))

with torch.no_grad():
    hf2(input_ids=input_ids, pixel_values=pixel_values, attention_mask=attention_mask,
        token_type_ids=torch.zeros_like(input_ids))
    model(input_ids, pixel_values, attention_mask)
for h in handles:
    h.remove()

print(f"{'block':22s} {'rel. error':>12s} {'max |diff|':>11s}")
prev = None
for name, _, _ in pairs:
    a, b = our_acts[name], hf_acts[name]
    if a.shape != b.shape:
        print(f"{name:22s} SHAPE DIFFERS: ours {tuple(a.shape)}, HF {tuple(b.shape)}")
        continue
    rel = ((a - b).norm() / b.norm()).item()
    flag = "  <<< jump" if prev is not None and rel > 3 * prev and rel > 2e-3 else ""
    print(f"{name:22s} {rel:12.2e} {(a - b).abs().max().item():11.3f}{flag}")
    prev = rel

### fp32 test: the same real weights without bf16 rounding

bf16 weights keep their exact values when converted to float32. We run the whole SigLIP (0.4 B parameters) and a few Gemma layers in **float32 on the CPU**
with both HF and our code and compare them. The Gemma layers get the HF intermediate outputs captured in the previous cell as input. If the difference is around 1e-5,
our code computes the same function as HF and any logit difference comes only from bf16.

In [ ]:
# fp32 test (CPU). If one test fails, the others still run
import copy
import paligemma


def fp32_cpu_copy(module):
    # To avoid an extra copy on the GPU, move to CPU, copy there, then move back
    module.to("cpu")
    copied = copy.deepcopy(module).float().eval()
    module.to(device)
    return copied


def report(name, a, b):
    a, b = as_tensor(a).float(), as_tensor(b).float()
    rel = ((a - b).norm() / b.norm()).item()
    print(f"[{'OK ' if rel < 1e-4 else '???'}] {name}: relative error {rel:.2e}, max |diff| {(a - b).abs().max().item():.2e}")


with torch.no_grad():
    try:
        px = pixel_values.float().cpu()
        report("SigLIP (27 layers, fp32)", fp32_cpu_copy(our_vt)(px), fp32_cpu_copy(hf_vt)(px))
    except Exception as e:
        print(f"SigLIP fp32 test failed to run: {type(e).__name__}: {e}")

    n_layers = len(our_lm.layers)
    hf_rot, our_rot = copy.deepcopy(first_attr(hf_lm, "rotary_emb")).cpu(), copy.deepcopy(our_lm.rotary_emb).cpu()
    pos = torch.arange(1, T + 1).unsqueeze(0)                          # PaliGemma positions start at 1
    mask = paligemma.make_prefix_lm_mask(T, T, torch.float32, torch.device("cpu"))
    for i in sorted({0, n_layers // 2, n_layers - 1}):
        try:
            x = (hf_acts["gemma_in0"] if i == 0 else hf_acts[f"Gemma layer {i - 1:2d}"]).float()   # HF's own intermediate output
            cos_h, sin_h = hf_rot(x, pos)
            cos_o, sin_o = our_rot(x, pos)
            out_hf = fp32_cpu_copy(hf_lm.layers[i])(x, attention_mask=mask, position_embeddings=(cos_h, sin_h))
            out_ours = fp32_cpu_copy(our_lm.layers[i])(x, (cos_o, sin_o), mask)
            report(f"Gemma layer {i:2d} (fp32)", out_ours, out_hf)
        except Exception as e:
            print(f"Gemma layer {i} fp32 test failed to run: {type(e).__name__}: {e}")

### Diagnosis: how do I read it?

| What you see | What it means |
|---|---|
| Disagreeing positions are mostly image positions, and HF's top1-top2 gap is small there | bf16 flips the order at unstable positions the model was never trained on. Harmless |
| The relative error grows smoothly layer by layer, no jump | bf16 accumulation |
| A jump at one block | Inspect that block's input/output, the cause is there. Example: dividing image features by `√d` and then multiplying by `√d` is a no-op mathematically, but in bf16 it adds two roundings and showed up as a jump at Gemma layer 0. HF does not scale the image at all and only multiplies the text by `√d`. Our code does the same now |
| fp32 tests all `OK` (~1e-5 or below) | The code computes the same function as HF with the real weights. What remains is only bf16 |
| A block shows `???` in the fp32 test | That block has a real difference, not bf16 |

In [ ]:
# Cleanup: remove the HF model from the GPU
# `pairs` holds references to the HF modules, so it must go too or the memory is not freed
del pairs, handles, hf2, hf_vt, hf_vt_in, hf_proj, hf_lm, hf_acts, our_acts, hf_rot, our_rot
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## KV cache

Two paths on the same input:
- `use_cache=False`: every step recomputes image + prompt + the answer so far, and SigLIP.
- `use_cache=True`: image + prompt are processed once (prefill), then every step feeds only the last token.

**Correctness:** both paths must match each other and the HF output from the beginning, token for token. In bf16 the cached and uncached logits
may not be bit-identical (matmuls of different shapes may sum in a different order). If two candidates are very close, a long generation may rarely
diverge at one token. If it does, we print the first differing position.

**Timing:** the GPU runs asynchronously, so we call `torch.cuda.synchronize()` before measuring. We also generate 64 tokens without eos (`eos_token_id=-1`)
and look at the time per step on both paths. How much the cache saves depends on the hardware: on a fast GPU at this sequence length the uncached step is
not much slower, because kernel launch overhead dominates; most of the saving then comes from SigLIP not running again at every step.
`checks/bench_kv_cache.py` measures the same thing locally.

In [ ]:
def timed_generate(use_cache, max_new_tokens, eos):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = generate.generate(model, input_ids, pixel_values, attention_mask, max_new_tokens, eos, use_cache=use_cache).cpu()
    torch.cuda.synchronize()
    return out, time.perf_counter() - t0


def first_mismatch(a, b):
    n = min(a.shape[1], b.shape[1])
    diff = (a[0, :n] != b[0, :n]).nonzero()
    return diff[0].item() if len(diff) else (None if a.shape[1] == b.shape[1] else n)


timed_generate(True, 2, eos_token_id)                  # warm-up: the first call loads the CUDA kernels

# 1) Correctness: same settings as the greedy generation (stops at eos)
cached, t_cached = timed_generate(True, MAX_NEW_TOKENS, eos_token_id)
uncached, t_uncached = timed_generate(False, MAX_NEW_TOKENS, eos_token_id)
print("cache    :", repr(hf_processor.decode(cached[0], skip_special_tokens=True)))
print("no cache :", repr(hf_processor.decode(uncached[0], skip_special_tokens=True)))
print("HF       :", repr(hf_processor.decode(ref_tokens[0], skip_special_tokens=True)))
for name, other in [("no cache", uncached), ("HF", ref_tokens)]:
    i = first_mismatch(cached, other)
    print(f"cache = {name}: {i is None}" + ("" if i is None else f" (first difference at token {i})"))

# 2) Timing: 64 tokens without eos, exactly 64 steps both ways
N = 64
long_cached, t_long_cached = timed_generate(True, N, -1)
long_uncached, t_long_uncached = timed_generate(False, N, -1)
print(f"\n{N} tokens, no eos:")
print(f"   no cache : {t_long_uncached:6.2f} s, {t_long_uncached / N * 1e3:6.1f} ms/token")
print(f"   cache    : {t_long_cached:6.2f} s, {t_long_cached / N * 1e3:6.1f} ms/token")
i = first_mismatch(long_cached, long_uncached)
print(f"   speedup {t_long_uncached / t_long_cached:.1f}x, identical tokens: {i is None}" + ("" if i is None else f" (first difference at token {i})"))

In [ ]:
# Time per step: the uncached step grows with the sequence length, the cached step stays nearly constant.
# The prefill (first step) is the same work both ways, the later steps diverge
from paligemma import KVCache

with torch.no_grad():
    for use_cache in (False, True):
        cache = KVCache() if use_cache else None
        ids, times = input_ids, []
        for step in range(N):
            torch.cuda.synchronize(); t0 = time.perf_counter()
            if step == 0 or not use_cache:
                logits = model(ids, pixel_values, torch.ones_like(ids), prefix_len=T, kv_cache=cache)
            else:
                logits = model(ids[:, -1:], None, torch.ones_like(ids[:, -1:]), kv_cache=cache)
            torch.cuda.synchronize(); times.append(time.perf_counter() - t0)
            ids = torch.cat([ids, logits[:, -1:].argmax(-1)], dim=1)
        picks = [0, 1, N // 4, N // 2, N - 1]
        print(f"{'cache' if use_cache else 'no cache':>9}: " + ", ".join(f"step {s}: {times[s] * 1e3:.0f} ms" for s in picks))